# Conversation State Management

## Purpose
Learn three approaches to managing multi-turn conversation history. Each approach has different trade-offs for control, convenience, and persistence, allowing you to choose the right pattern for your use case.

## Key Concepts
- **Client-Side State**: You manage conversation history manually
- **Server-Side Sessions**: Automatic persistence with SQLiteSession
- **Response ID Tracking**: Server-managed state via response IDs
- **Trade-offs**: Control vs convenience, stateless vs stateful

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)

## Import Libraries

Import session management tools:

In [ ]:
import asyncio
from agents import Agent, Runner, SQLiteSession

## Create Test Agent

Simple agent for demonstrating state management:

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=model_id
)

## Approach 1: Client-Side Input List

You manually manage conversation history using `result.to_input_list()`.

**How It Works**:
1. Run agent with first message
2. Convert result to input list: `result.to_input_list()`
3. Append new user message to list
4. Pass entire list to next `Runner.run()`

**Pros**:
- Full control over conversation history
- No server dependencies
- Can edit/filter history before next turn
- Stateless - easy to test and debug

**Cons**:
- Must manage state yourself
- Need to store history between sessions
- More boilerplate code

💡 **Use When**: You need full control or want stateless architecture.

In [ ]:
# First message
result = await Runner.run(agent, "My name is Alice.")
print("Turn 1:", result.final_output)

# Convert to input list and add new message
conversation_history = result.to_input_list()
conversation_history.append({"role": "user", "content": "What is my name?"})

# Second message with history
result = await Runner.run(agent, conversation_history)
print("Turn 2:", result.final_output)

## Approach 2: Server-Side Sessions

Automatic history persistence using `SQLiteSession`.

**How It Works**:
1. Create session with unique `thread_id`
2. Pass session to `Runner.run()`
3. History automatically saved and loaded
4. Each thread_id maintains separate conversation

**Pros**:
- Automatic state management
- Persistent storage across restarts
- Simple API - just pass session object
- Built-in SQLite backend

**Cons**:
- Requires session storage setup
- Less control over history
- Database overhead

💡 **Use When**: You want automatic persistence and don't need fine-grained control.

In [ ]:
# Create a session with a unique thread ID
session = SQLiteSession("user_alice_123")

In [ ]:
# First message
result = await Runner.run(agent, "My name is Alice.", session=session)
print("Turn 1:", result.final_output)

# Second message - session automatically maintains context
result = await Runner.run(agent, "What is my name?", session=session)
print("Turn 2:", result.final_output)

## Approach 3: Response ID Tracking

Server-side state using response IDs without explicit session objects.

**How It Works**:
1. Run agent, capture `result.last_response_id`
2. Pass `previous_response_id` to next `Runner.run()`
3. Server maintains state via response ID chain

**Alternative**: Use `auto_previous_response_id=True` for automatic tracking

**Pros**:
- Simpler than sessions
- Server-managed state
- No session object to maintain

**Cons**:
- Must track response IDs
- Less explicit than other approaches
- Relies on server-side storage

💡 **Use When**: You want server-managed state without session complexity.

In [ ]:
# First message
result = await Runner.run(agent, "My name is Bob.")
print("Turn 1:", result.final_output)

# Capture response ID for next call
last_response_id = result.last_response_id

# Second message with response ID
result = await Runner.run(
    agent, 
    "What is my name?",
    previous_response_id=last_response_id
)
print("Turn 2:", result.final_output)

### Alternative: Auto-Track Response ID

Use `auto_previous_response_id=True` to automatically track without manual ID management:

In [ ]:
# Alternative: Auto-track response ID
result = await Runner.run(
    agent,
    "My name is Charlie.",
    auto_previous_response_id=True
)
print("With auto tracking:", result.final_output)

## 🎉 Congratulations!

You've completed the **Conversation State Management** notebook!